## Continuing the learning journey

In [notebook 04](04-prepare-message-classification-dataset.ipynb), we prepared labelled music-service messages for training and evaluation. Here we use the 12 evaluation examples to establish the baseline for the planned notebook 06 on fine tuning.

**Reading guide:** follow the explanations and short examples first. Supporting checks and reporting code are folded where the viewer supports it. Run every code cell in order, including folded cells; understanding every implementation detail is optional.


## The question

**How do MLX-LM and PyTorch MPS compare when the same Qwen model classifies the same messages?**

We will compare prediction quality and timing, then inspect the mistakes to understand what needs improvement.


## Why use both workflows?

MLX-LM connects the data format introduced in notebook 04 to actual predictions. PyTorch introduces our chosen fine-tuning workflow. Comparing them shows how the same classification task carries across frameworks.

| Workflow | How it runs the model |
|---|---|
| MLX-LM with MLX | MLX-LM loads the language model and manages generation, while MLX performs the numerical computation. |
| Transformers with PyTorch MPS | Transformers loads the language model and manages generation, while PyTorch uses its MPS backend to compute on the Mac GPU. |

A **backend** carries out computation on particular hardware. MLX-LM is a language-model package, whereas MPS is a PyTorch backend, so we compare complete inference workflows rather than equivalent software components.

Timing results do not determine our fine-tuning framework. Run both workflows because the comparison cells need both sets of results.

**Where does the model run?** Both workflows load Qwen into the notebook's Python process. Files come from Hugging Face and are cached on your Mac; generation then runs locally. No Ollama service or HTTP model server is needed. VS Code is the editor in which we run the notebook.

Sources: [MLX-LM](https://github.com/ml-explore/mlx-lm) and [PyTorch MPS](https://docs.pytorch.org/docs/stable/notes/mps.html).


## What inference means in this lesson

**Inference means using a trained model to make a prediction.** We use [Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct), a small language model with about half a billion parameters. Both frameworks run this same model.

The `Instruct` checkpoint has already been trained to follow instructions. Here, “original model” means that checkpoint before any additional training on our music service dataset. It receives a message and generates the category that should handle it.

```text
Message: Can you recommend something similar to this song?
Expected category: Recommendation_Request
```

The six possible categories are `Recommendation_Request`, `Explanation_Request`, `Customer_Support`, `Partnership`, `Junk`, and `Human_Review`.

To obtain a prediction, we format the conversation, convert it into tokens, and pass those tokens through the model. The model generates its answer one token at a time, and we convert the output back into text. We then compare that text with the expected category.

The model's learned weights stay unchanged throughout this process. **Inference uses those weights, evaluation checks the predictions, and fine tuning changes the weights through additional training.** Our evaluation of the unchanged model becomes the classification baseline.

## Keep the comparison consistent

Both workflows use the same Qwen revision, messages, instructions, input token IDs, and generation settings. In this experiment, both store the weights as **bfloat16**, a numerical format that uses 16 bits per value. We process one message at a time and release each model before loading the other.

We compare:

- **Accuracy:** the percentage of messages assigned the correct category.
- **Agreement:** how often both workflows return the same category.
- **Median latency:** the middle request duration across repeated measurements.
- **Model loading time:** how long it takes to load each model from the local files, measured separately from request latency.

Two workflows can agree and still be wrong, so we will also inspect their mistakes.


## Prepare the environment

**Platform scope:** this notebook runs as written on an Apple Silicon Mac because it compares MLX-LM with PyTorch's MPS backend. The dataset, prompts, tokenization, and scoring approach also apply to other machines. On Windows or Linux, the PyTorch path could be adapted to use CUDA on a supported NVIDIA GPU or to run on the CPU. Device selection, timing, and memory cleanup would need corresponding changes. That would be a different runtime comparison, so the Mac timings reported here would not carry over.

This notebook uses a separate Python environment because the pinned MLX-LM version requires Transformers 5, while `sentence-transformers==5.1.2` in the earlier lessons requires Transformers below version 5.

From the repository root, run:

```bash
python3.11 -m venv .venv-inference
source .venv-inference/bin/activate
pip install -r requirements-mlx.txt torch==2.14.1 transformers==5.22.1 nbformat==5.10.4 ipykernel==6.29.4
```

Select `.venv-inference/bin/python` as the notebook environment, restart the kernel, and run the cells in order on an Apple Silicon Mac. Continue using `.venv` for the earlier embedding lessons.

The first run may download model files. Predictions run locally. Save the notebook after execution to retain its comparison outputs as the baseline for the next lesson. Pause other model workloads before collecting timings to reduce competition for the GPU.

## Step 1: Inspect an evaluation example

We begin with `evaluation.jsonl` from notebook 04. Each record contains a conversation in `messages` and its expected answer in `label`.

The conversation has a system message explaining the task and a user message containing the text to classify. We send only these messages to the model and keep the expected answer separate for scoring. Including the answer in the conversation would give it away.

The next cell loads all 12 records into `evaluation_records`, a Python list of dictionaries, then selects the first with `[0]`. Selecting one example does not discard the others. The cell works from either the repository root or the notebook directory and does not load a model.

The records have this structure, with the instruction shortened for display:

```python
evaluation_records = [
    {
        "messages": [
            {"role": "system", "content": "You route messages sent to a music service..."},
            {"role": "user", "content": "Classify: Can you recommend music similar to my favourites?"},
        ],
        "label": "Recommendation_Request",
    },
    # The remaining evaluation records follow the same structure.
]
```


In [34]:
import json
from pathlib import Path

notebook_dir = Path.cwd()
if not (notebook_dir / "data/music_service_intents_mlx/evaluation.jsonl").is_file():
    notebook_dir = notebook_dir / "embeddings-and-search"

evaluation_path = notebook_dir / "data/music_service_intents_mlx/evaluation.jsonl"
if not evaluation_path.is_file():
    raise FileNotFoundError("Run notebook 04 first, then open this notebook from its folder or the repository root.")

with evaluation_path.open(encoding="utf-8") as file:
    evaluation_records = [json.loads(line) for line in file if line.strip()]
if not evaluation_records:
    raise ValueError("The evaluation file contains no examples.")

example = evaluation_records[0]
model_messages = example["messages"]
expected_label = example["label"]

print(f"Loaded {len(evaluation_records)} evaluation examples.")
print("\nWhat the model will see:")
for message in model_messages:
    print(f"{message['role']}: {message['content']}")
print(f"\nExpected answer (kept outside the model input): {expected_label}")

Loaded 12 evaluation examples.

What the model will see:
system: You route messages sent to a music service.
Classify each message as Recommendation_Request, Explanation_Request, Customer_Support, Partnership, Junk, or Human_Review.
Use Human_Review when the intent is unclear or more than one route is equally plausible.
Return only the category name, without an explanation.
user: Classify: Can you recommend music similar to the tracks in my favourites?

Expected answer (kept outside the model input): Recommendation_Request


### Understand what we have loaded

The output shows the **input** that the model will receive and the **expected label** that we will use to check its answer. The **prediction** does not exist yet because we have not run inference.

Before continuing, read the user message and choose a category yourself. Then compare it with the expected label and check that `model_messages` contains only the system and user messages.

We now have a conversation, but Qwen needs a sequence of token IDs to process it.

## Step 2: Prepare the model input

Preparing the input involves two operations: formatting the conversation and converting the formatted text into token IDs.

A **chat template** adds markers that identify the speaker and the boundaries between messages. We use Qwen's own template so the conversation follows the format it learned during training.

A **tokenizer** splits that text into tokens and looks up their integer IDs in a vocabulary. A token may represent a word, part of a word, punctuation, or a special marker. The ID identifies a vocabulary entry; later, the model uses it to look up an embedding vector.

The cell below loads Qwen's tokenizer and pins its revision so both workflows use the same version. We first display the formatted conversation, then request its token IDs with `tokenize=True`. Setting `return_dict=False` gives us a plain Python list, while `add_generation_prompt=True` adds the start of the assistant's turn.

Although `AutoTokenizer` comes from Transformers, this step does not run PyTorch inference. The prepared IDs will be passed directly to both workflows.

Reference: [Hugging Face chat templates](https://huggingface.co/docs/transformers/chat_templating).

**What is a model revision?** The model ID identifies its repository, while the revision identifies a particular version of the files, like a Git commit. Pinning it ensures that both workflows load the same weights and tokenizer even if the repository is updated later.

In [35]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)

formatted_prompt = tokenizer.apply_chat_template(
    model_messages, tokenize=False, add_generation_prompt=True
)
input_ids = tokenizer.apply_chat_template(
    model_messages, tokenize=True, add_generation_prompt=True, return_dict=False
)

print("Formatted conversation:")
print(formatted_prompt)
print(f"Input token count: {len(input_ids)}")

Formatted conversation:
<|im_start|>system
You route messages sent to a music service.
Classify each message as Recommendation_Request, Explanation_Request, Customer_Support, Partnership, Junk, or Human_Review.
Use Human_Review when the intent is unclear or more than one route is equally plausible.
Return only the category name, without an explanation.<|im_end|>
<|im_start|>user
Classify: Can you recommend music similar to the tracks in my favourites?<|im_end|>
<|im_start|>assistant

Input token count: 89


### Read the formatted conversation

`<|im_start|>` marks the beginning of a message, and `<|im_end|>` marks its end. The text following the start marker identifies the speaker. At the bottom, the assistant header marks where the model should begin its answer.

The token count includes the instructions, user message, and conversation markers.

**Try it:** set `add_generation_prompt=False` in both calls and inspect what disappears. This changes where the model is expected to continue, rather than choosing a category for it. Restore `True` before running the remaining cells.

With the input format understood, we can choose the generation settings and prepare the full evaluation.

## Step 3: Choose how the model generates answers

The input is ready, so we can load the model's **weights**, which are the numerical values learned during training. Before running it, we need to choose how it will select the tokens that form its answer.

### How greedy decoding builds an answer

A language model builds its answer one token at a time. At each step, it assigns a probability to every possible next token, based on the input and the tokens it has already generated. **Decoding is the strategy used to choose which token comes next.**

With **greedy decoding**, the model always chooses the token with the highest probability at that step. It is called greedy because it takes the best available choice immediately, without looking ahead to compare complete answers.

Imagine the model assigns these probabilities to the next token:

| Possible next token | Illustrative probability |
|---|---:|
| `Recommendation` | 60% |
| `Human` | 30% |
| All other tokens combined | 10% |

Greedy decoding chooses `Recommendation`. With that token added to the answer, the model calculates the probabilities again and chooses the next token, which might be `_Request`. This process continues until the model produces a stopping token or reaches the output limit.

These numbers are an illustration rather than measurements from our experiment. They represent probabilities for the next token, so 60% for `Recommendation` does not mean the model is 60% confident that the complete category is correct.

### Why we use it here

We want to compare MLX-LM and PyTorch MPS under similar conditions. Random sampling can select different tokens across repeated requests, so we use greedy decoding to remove that source of variation.

The two frameworks express this choice differently:

- MLX-LM uses `make_sampler(temp=0.0)`.
- PyTorch uses `do_sample=False` with `num_beams=1`.

Both settings select the most probable next token, although numerical differences between the backends can still lead to different predictions. Making the most likely choice at every step also does not guarantee the correct category or the most likely complete answer.

Finally, distinguish this selection process from the tokenizer's `decode()` method. **Greedy decoding chooses the output tokens; `tokenizer.decode()` converts their IDs into readable text.**

### Shared setup

The next cell prepares the settings and downloads the pinned checkpoint if needed, which requires roughly 1 GB of storage. Both backends will use these same local files and a limit of 32 new tokens per answer.

Reference: [MLX-LM Python API](https://github.com/ml-explore/mlx-lm#python-api).

In [36]:
from importlib.metadata import version
from time import perf_counter

from huggingface_hub import snapshot_download

MAX_NEW_TOKENS = 32
valid_labels = {
    "Recommendation_Request", "Explanation_Request", "Customer_Support",
    "Partnership", "Junk", "Human_Review",
}
# Fetch files before timing either model load.
model_path = snapshot_download(
    MODEL_ID, revision=MODEL_REVISION,
    allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.jinja"],
)
print("Model revision:", MODEL_REVISION)
print("Maximum new tokens per answer:", MAX_NEW_TOKENS)

Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 3325.42it/s]

Model revision: 7ae557604adf67be50417f59c2c2f167def9a775
Maximum new tokens per answer: 32


## Step 4: Evaluate all 12 messages

Both backends use one shared evaluator, `evaluate_backend()`, so their predictions are scored and timed in the same way:

1. Run one **warm up** request before timing to reduce the influence of first-use setup.
2. Loop over all 12 messages, generating and scoring one answer at a time.
3. Repeat the loop three times to observe timing variation.

That gives **12 messages × 3 passes × 2 backends = 72 measured requests**, plus one untimed warm up per backend. Repeated passes provide more timing samples; accuracy still describes only 12 distinct examples.

We accept only an exact category name after removing surrounding whitespace. A valid category can still be incorrect: `Junk`, for example, would be wrong for a recommendation request. Explanations and unknown labels count as invalid outputs.

### Shared evaluation behavior

Each workflow defines a prediction function that receives prepared token IDs and returns the answer text, generated token count, and stopping reason. Defining the function does not run predictions; the evaluator calls it for each message and records correctness and duration alongside those details.

Both models use **evaluation mode**, so any layers with different training behavior use their inference behavior. **Synchronization** waits for submitted device work to finish, so timings include completed computation. A fresh generation cache is created for each request, keeping messages independent.

The supporting cell prepares all input IDs and defines the evaluator. You can run it without studying every check on your first reading.


In [37]:
#| code-fold: true
#| code-summary: "Show supporting code"
import gc

import pandas as pd
from IPython.display import display

REPEATS = 3

def score_prediction(raw, expected):
    prediction = raw.strip()
    return prediction, prediction in valid_labels, prediction == expected

prepared_inputs = []
for record in evaluation_records:
    ids = tokenizer.apply_chat_template(
        record["messages"], tokenize=True, add_generation_prompt=True, return_dict=False
    )
    prepared_inputs.append(ids)

def evaluate_backend(name, predict, synchronize):
    predict(prepared_inputs[0])  # Warm-up is excluded from measured passes.
    rows = []
    for trial in range(1, REPEATS + 1):
        for example_id, (record, ids) in enumerate(zip(evaluation_records, prepared_inputs)):
            synchronize()
            start = perf_counter()
            raw, token_count, finish_reason = predict(ids)
            synchronize()
            elapsed = perf_counter() - start
            prediction, valid, correct = score_prediction(raw, record["label"])
            rows.append({
                "backend": name, "trial": trial, "example_id": example_id,
                "expected": record["label"], "raw_prediction": raw,
                "prediction": prediction, "valid": valid, "correct": correct,
                "input_tokens": len(ids), "generated_tokens": token_count,
                "finish_reason": finish_reason, "latency_s": elapsed,
            })
        print(f"{name}: completed pass {trial}/{REPEATS}")
    return rows

print(f"Prepared {len(prepared_inputs)} evaluation inputs.")

Prepared 12 evaluation inputs.


### Run the MLX-LM evaluation

**What:** Run the evaluation with MLX-LM and store the results in `mlx_rows`.

**Why:** Establish the first set of predictions and timings for the comparison.

**How:**

1. **Load and prepare.** Load the local model and tokenizer, measure loading time with synchronization, and select evaluation mode.
2. **Share the stopping rules.** Models normally provide default end-of-answer token IDs. Here we explicitly pass MLX-LM's IDs to PyTorch so differences in stopping rules do not affect the comparison.
3. **Define one prediction.** `predict_mlx(ids)` uses `stream_generate()` to produce text incrementally and collects the pieces into one answer. The sampler chooses each next token; `temp=0.0` selects the greedy decoding introduced in Step 3.
4. **Evaluate and release.** Pass the function to the shared evaluator, then delete the model and tokenizer and clear unused memory before loading PyTorch.

```text
Prepared token IDs → predict_mlx() → Answer, token count, stopping reason
                         ↓
                 Shared evaluator → mlx_rows
```


In [38]:
import mlx.core as mx
from mlx_lm import load, stream_generate
from mlx_lm.sample_utils import make_sampler

mx.synchronize()
load_start = perf_counter()
mlx_model, mlx_tokenizer = load(model_path)
mx.synchronize()
mlx_load_seconds = perf_counter() - load_start
mlx_model.eval()

STOP_TOKEN_IDS = sorted(mlx_tokenizer.eos_token_ids)
print("MLX-LM:", version("mlx-lm"), "MLX:", version("mlx"))
print(f"Local model loading: {mlx_load_seconds:.3f} seconds")

def predict_mlx(ids):
    pieces = []
    for response in stream_generate(
        mlx_model, mlx_tokenizer, prompt=ids,
        max_tokens=MAX_NEW_TOKENS, sampler=make_sampler(temp=0.0),
    ):
        pieces.append(response.text)
    return "".join(pieces), response.generation_tokens, response.finish_reason

mlx_rows = evaluate_backend("MLX-LM", predict_mlx, mx.synchronize)
# Release weights before loading the second backend.
mx.synchronize()
del mlx_model, mlx_tokenizer
gc.collect()
mx.clear_cache()
print("MLX model released.")

MLX-LM: 0.32.2 MLX: 0.32.4
Local model loading: 0.366 seconds
MLX-LM: completed pass 1/3
MLX-LM: completed pass 2/3
MLX-LM: completed pass 3/3
MLX model released.


### Run the PyTorch MPS evaluation

**What:** Repeat the evaluation using Transformers and PyTorch on the Mac GPU. Store the results in `torch_rows`.

**Why:** Compare this workflow with MLX-LM and establish the PyTorch baseline for fine tuning.

**How:**

1. **Check and load.** Confirm that MPS is available and operations will not automatically fall back to the CPU. Load the local model onto the GPU in bfloat16, measure loading time, and select evaluation mode.
2. **Set the generation rules.** Configure greedy decoding, the shared output-token limit, and MLX-LM's end-of-answer token IDs.
3. **Define one prediction.** `predict_torch(ids)` generates an answer without tracking the information needed for training. For this model, `generate()` returns the input followed by the answer, so the function keeps only the new tokens and converts them into text.
4. **Evaluate and release.** Pass the function to the shared evaluator, then delete the model and clear unused GPU memory.

```text
Prepared token IDs → predict_torch() → Answer, token count, stopping reason
                         ↓
                 Shared evaluator → torch_rows
```

**From input IDs to answer text:** for this decoder-only model, `generate()` returns the original input IDs followed by the newly generated IDs. Using invented IDs:

```text
Input ids:       [10, 20, 30]
Full output:    [[10, 20, 30, 501, 502, 99]]
Answer IDs:                 [501, 502, 99]
Decoded answer: "Customer_Support"
```

The outer brackets in `output` represent a batch containing one sequence. `output[0, len(ids):]` selects that sequence and skips its three input positions; `tokenizer.decode(..., skip_special_tokens=True)` converts the remaining IDs to text while omitting special markers such as the stopping token (illustrated by `99`). The boundary comes from the input length, not the token values.


In [39]:
import os
import torch
from transformers import AutoModelForCausalLM, GenerationConfig

if not torch.backends.mps.is_available():
    raise RuntimeError("MPS is unavailable. Use an Apple Silicon Mac with an MPS capable PyTorch installation.")
if os.environ.get("PYTORCH_ENABLE_MPS_FALLBACK") == "1":
    raise RuntimeError("Restart the kernel without MPS CPU fallback for this comparison.")

torch.mps.synchronize()
load_start = perf_counter()
torch_model = AutoModelForCausalLM.from_pretrained(
    model_path, dtype=torch.bfloat16, local_files_only=True,
    attn_implementation="sdpa",
).to("mps")
torch_model.eval()
torch.mps.synchronize()
torch_load_seconds = perf_counter() - load_start
torch_weight_types = sorted({str(p.dtype) for p in torch_model.parameters()})

generation_config = GenerationConfig(
    do_sample=False, num_beams=1, max_new_tokens=MAX_NEW_TOKENS,
    eos_token_id=STOP_TOKEN_IDS, pad_token_id=tokenizer.pad_token_id,
    bos_token_id=tokenizer.bos_token_id, use_cache=True,
)

def predict_torch(ids):
    inputs = torch.tensor([ids], dtype=torch.long, device="mps")
    with torch.inference_mode():
        output = torch_model.generate(
            input_ids=inputs, attention_mask=torch.ones_like(inputs),
            generation_config=generation_config,
        )
    generated_ids = output[0, len(ids):].tolist()
    raw = tokenizer.decode(generated_ids, skip_special_tokens=True)
    stopped = bool(generated_ids) and generated_ids[-1] in STOP_TOKEN_IDS
    return raw, len(generated_ids), "stop" if stopped else "length"

print("PyTorch:", version("torch"), "Transformers:", version("transformers"))
print("Device:", next(torch_model.parameters()).device)
print("Weight types:", torch_weight_types)
print(f"Local model loading: {torch_load_seconds:.3f} seconds")

torch_rows = evaluate_backend("PyTorch MPS", predict_torch, torch.mps.synchronize)
torch.mps.synchronize()
del torch_model
gc.collect()
torch.mps.empty_cache()
print("PyTorch model released.")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 27212.59it/s]


PyTorch: 2.14.1 Transformers: 5.22.1
Device: mps:0
Weight types: ['torch.bfloat16']
Local model loading: 0.254 seconds
PyTorch MPS: completed pass 1/3
PyTorch MPS: completed pass 2/3
PyTorch MPS: completed pass 3/3
PyTorch model released.


## Step 5: Read the comparison

The summary table reports accuracy from the first measured pass and latency across all three passes, in milliseconds. Model loading times were printed in the evaluation cells and are recorded separately in `summary`. The agreement result compares the two workflows' predictions from the first pass.

The error table shows incorrectly classified messages alongside their expected labels and raw predictions. Use it to identify which categories need improvement.


In [40]:
#| code-fold: true
#| code-summary: "Show supporting code"
results = pd.DataFrame(mlx_rows + torch_rows)
first_pass = results[results["trial"] == 1]
load_seconds = {"MLX-LM": mlx_load_seconds, "PyTorch MPS": torch_load_seconds}
summary_rows = []
for backend, group in results.groupby("backend", sort=False):
    first = group[group["trial"] == 1]
    pass_medians = group.groupby("trial")["latency_s"].median()
    changed = int((group.groupby("example_id")["raw_prediction"].nunique() > 1).sum())
    summary_rows.append({
        "backend": backend, "correct": int(first["correct"].sum()), "examples": len(first),
        "accuracy_pct": 100 * first["correct"].mean(),
        "invalid_outputs": int((~first["valid"]).sum()),
        "local_load_s": load_seconds[backend],
        "median_latency_s": group["latency_s"].median(),
        "p90_latency_s": group["latency_s"].quantile(0.9),
        "pass_median_min_s": pass_medians.min(), "pass_median_max_s": pass_medians.max(),
        "median_generated_tokens": group["generated_tokens"].median(),
        "outputs_changed_across_passes": changed,
    })
summary = pd.DataFrame(summary_rows)

comparison = first_pass.pivot(index="example_id", columns="backend", values="prediction")
comparison.insert(0, "expected", [r["label"] for r in evaluation_records])
comparison["agree"] = comparison["MLX-LM"] == comparison["PyTorch MPS"]
agreement = float(comparison["agree"].mean())
errors = first_pass[~first_pass["correct"]].copy()
errors["message"] = [evaluation_records[i]["messages"][1]["content"] for i in errors["example_id"]]

In [41]:
overview = summary[
    ["backend", "correct", "examples", "accuracy_pct"]
].copy()

overview["median_latency_ms"] = summary["median_latency_s"] * 1000
display(overview.round(2))

print(f"Both backends agreed on {agreement:.0%} of messages.")

print("\nIncorrect predictions:")
with pd.option_context("display.max_colwidth", None):
    display(errors[["backend", "message", "expected", "raw_prediction"]])

,backend,correct,examples,accuracy_pct,median_latency_ms
0,MLX-LM,4,12,33.33,32.09
1,PyTorch MPS,4,12,33.33,36.94


Both backends agreed on 100% of messages.

Incorrect predictions:


,backend,message,expected,raw_prediction
1,MLX-LM,Classify: Can our festival partner with your recommendation platform?,Partnership,Human_Review
2,MLX-LM,Classify: You have been selected for an exclusive cash bonus.,Junk,Human_Review
3,MLX-LM,Classify: My subscription payment was charged twice.,Customer_Support,Human_Review
4,MLX-LM,Classify: I cannot remove a track from my library.,Customer_Support,Human_Review
5,MLX-LM,Classify: Congratulations you won a prize click here now.,Junk,Human_Review
6,MLX-LM,Classify: Why did you recommend this song to me?,Explanation_Request,Human_Review
8,MLX-LM,Classify: We would like to integrate your service into our music app.,Partnership,Human_Review
10,MLX-LM,Classify: Help me understand this recommendation.,Explanation_Request,Human_Review
37,PyTorch MPS,Classify: Can our festival partner with your recommendation platform?,Partnership,Human_Review
38,PyTorch MPS,Classify: You have been selected for an exclusive cash bonus.,Junk,Human_Review


## What we learned and what comes next

Read accuracy and agreement together. If both workflows produce the same incorrect answer, the immediate question is how the model and prompt handle the task, rather than which runtime is correct. Inspect which categories are being confused before deciding what might help.

Our dataset has only two examples per category, so one mistake changes accuracy by about 8.3 percentage points. It supports learning and error analysis, but cannot establish production reliability.

The timings describe this Mac and these short requests. They include input conversion, generation, decoding, and completion of GPU work, but exclude downloads and tokenization. We ran MLX first and PyTorch second, so caching and background activity may affect the results. These measurements do not establish which workflow is faster across other workloads.

### Why fine tuning is the next step

In the saved run, the baseline falls short of the classification quality we want: both workflows correctly classified only **4 of 12 messages**, and both routed most messages to `Human_Review`. The eight mistakes all come from partnership, junk, customer support, and explanation requests being routed to `Human_Review`. Agreement shows that both workflows made the same choices on this small set; it does not establish general equivalence between the frameworks.

Our next step is therefore to **test fine tuning with PyTorch MPS**. We will give the model labelled training examples to help it learn the distinctions between our six categories, choose training settings using validation data, and evaluate it through the same PyTorch inference path on the unchanged test set.

Fine tuning is an experiment rather than a guaranteed solution. Clearer instructions or better examples may also help, and the small test set limits what we can conclude. The comparison results give us a concrete question for notebook 06: **does additional training reduce these classification errors?** Use the training and validation splits to choose improvements, and keep these test messages fixed for the final comparison. Training memory requirements and settings will be checked as part of that lesson.

Three ideas to retain:

1. Tokenization prepares input, and the inference backend computes predictions.
2. Accuracy measures correctness, while agreement compares the two workflows.
3. A baseline records the starting point for measuring improvement.
